In [1]:
import pandas as pd
import scanpy as sc
import os
from scipy.sparse import issparse

In [2]:
results = sc.read('../predict.h5ad')

In [4]:
results.obs.to_csv('results.csv')

In [2]:
data = pd.read_csv('ig_score_changes.csv')

In [3]:
data

,Gene,IG Score Before Training,IG Score After Training
0,C1orf141,0.268941,0.000243
1,PKP1,0.268941,0.000022
2,RERE,0.268941,0.000227
3,CSMD2,0.268941,0.000242
4,HIVEP3,0.268941,0.000245
...,...,...,...
27195,ANKRD20A12P,0.268941,0.268941
27196,MAFIP,0.268941,0.268941
27197,MGC70870,0.268941,0.268941
27198,LOC102724728,0.268941,0.268941


In [5]:
data['change'] = data['IG Score Before Training'] - data['IG Score After Training']

In [6]:
data

,Gene,IG Score Before Training,IG Score After Training,change
0,C1orf141,0.268941,0.000243,0.268699
1,PKP1,0.268941,0.000022,0.268919
2,RERE,0.268941,0.000227,0.268714
3,CSMD2,0.268941,0.000242,0.268700
4,HIVEP3,0.268941,0.000245,0.268696
...,...,...,...,...
27195,ANKRD20A12P,0.268941,0.268941,0.000000
27196,MAFIP,0.268941,0.268941,0.000000
27197,MGC70870,0.268941,0.268941,0.000000
27198,LOC102724728,0.268941,0.268941,0.000000


In [7]:
#sort by change
data.sort_values('change', ascending=False)

,Gene,IG Score Before Training,IG Score After Training,change
14605,VIM,0.268941,0.000005,0.268937
17412,YBX3,0.268941,0.000014,0.268928
1840,MCL1,0.268941,0.000015,0.268927
23186,LGALS3BP,0.268941,0.000015,0.268927
25274,LENG8,0.268941,0.000015,0.268927
...,...,...,...,...
25970,WFDC2,0.268941,0.954725,-0.685784
24846,APOE,0.268941,0.973593,-0.704652
13012,TMSB4X,0.268941,0.979171,-0.710229
3724,SFTPB,0.268941,0.982532,-0.713591


In [9]:
data.to_csv('change.csv',index=False)

In [7]:
import os
def find_T_cell_h5ad(root_dir):
    T_cell_paths = []
    for dirpath, _, filenames in os.walk(root_dir):
        if 'T_cell.h5ad' in filenames:
            T_cell_paths.append(os.path.join(dirpath, 'Neutrophil.h5ad'))
    
    return T_cell_paths

def main():
    root_dir = '/project/DPDS/Wang_lab/s439765/spatial_tcr/spatial_transcriptomics/merscope_data'  
    T_cell_paths = find_T_cell_h5ad(root_dir)
    
    for path in T_cell_paths:
        print(path)

if __name__ == "__main__":
    main()

/project/DPDS/Wang_lab/s439765/spatial_tcr/spatial_transcriptomics/merscope_data/HumanLiverCancerPatient1/Neutrophil.h5ad
/project/DPDS/Wang_lab/s439765/spatial_tcr/spatial_transcriptomics/merscope_data/HumanMelanomaPatient1/Neutrophil.h5ad
/project/DPDS/Wang_lab/s439765/spatial_tcr/spatial_transcriptomics/merscope_data/HumanProstateCancerPatient2/Neutrophil.h5ad
/project/DPDS/Wang_lab/s439765/spatial_tcr/spatial_transcriptomics/merscope_data/HumanLungCancerPatient2/Neutrophil.h5ad
/project/DPDS/Wang_lab/s439765/spatial_tcr/spatial_transcriptomics/merscope_data/HumanOvarianCancerPatient1/Neutrophil.h5ad
/project/DPDS/Wang_lab/s439765/spatial_tcr/spatial_transcriptomics/merscope_data/HumanMelanomaPatient2/Neutrophil.h5ad
/project/DPDS/Wang_lab/s439765/spatial_tcr/spatial_transcriptomics/merscope_data/HumanOvarianCancerPatient2Slice1/Neutrophil.h5ad
/project/DPDS/Wang_lab/s439765/spatial_tcr/spatial_transcriptomics/merscope_data/HumanColonCancerPatient1/Neutrophil.h5ad
/project/DPDS/Wang

In [3]:
def preprocess_data(adata, immune_cell, n_genes):
    # Read the data
    if immune_cell == 'tcell':
        immune_cell = 'T'
    elif immune_cell == 'bcell':
        immune_cell = 'B'
    else:
        raise ValueError('Invalid immune cell type')

    # Ensure adata is not a view
    adata = adata.copy()

    # Filter the tumor cells
    tumor_cells = adata[adata.obs['cell_type'].astype(int) == 1].copy()

    # Check if the expression matrix is sparse and convert to dense if necessary
    if issparse(tumor_cells.X):
        tumor_cells_X_dense = tumor_cells.X.toarray()
    else:
        tumor_cells_X_dense = tumor_cells.X

    # Calculate mean expression
    mean_expression = tumor_cells_X_dense.mean(axis=0)

    # Select top n genes
    top_n_genes = mean_expression.argsort()[-n_genes:][::-1]
    adata = adata[:, top_n_genes].copy()
    adata.obs[immune_cell] = adata.obs[immune_cell].astype(float)
    tumor_cells.obs[immune_cell] = tumor_cells.obs[immune_cell].astype(float)
    # Calculate the 50th percentile of the immune cell column
    percentile_value = np.percentile(tumor_cells.obs[immune_cell], 50)

    # Binarize the immune cell column based on the percentile value
    adata.obs[immune_cell] = np.where(adata.obs[immune_cell] >= percentile_value, 1, 0)

    return adata

In [24]:
adata = sc.read_h5ad('/project/DPDS/Wang_lab/shared/spatial_TCR/data/train_validate/Spatial_TCR_Visium/p16/p16.h5ad')

In [17]:
adata.obs['T'].unique()

array([0, 1])

In [25]:
adata = sc.read_h5ad('/project/DPDS/Wang_lab/shared/spatial_TCR/data/train_validate/Visium/HumanBreastCancerWholeTranscriptome/T_cell.h5ad')

In [27]:
print("Unique cell_type values:", adata.obs['cell_type'].unique())

Unique cell_type values: ['1', '0']
Categories (2, object): ['0', '1']


In [21]:
tumor_cells = adata[adata.obs['cell_type'].astype(int) == 1].copy()

In [22]:
 if issparse(adata.X):
    tumor_cells_X_dense = tumor_cells.X.toarray()
else:
    tumor_cells_X_dense = tumor_cells.X



In [23]:
mean_expression = tumor_cells_X_dense.mean(axis=0)

In [2]:
import pandas as pd

In [213]:
df1 = pd.read_csv('filtered_genes_500.csv')
df2 = pd.read_csv('top500.csv')

In [5]:
df2.dropna( inplace=True)

In [210]:
df2

,Unnamed: 0,gene_ids
0,IGHM,ENSG00000211899
1,IGLC1,ENSG00000211675
2,IGKC,ENSG00000211592
3,JCHAIN,ENSG00000132465
4,B2M,ENSG00000166710
...,...,...
495,CTNNB1,ENSG00000168036
496,PTGS1,ENSG00000095303
497,CLTC,ENSG00000141367
498,HP1BP3,ENSG00000127483


In [218]:
common_genes = set(df1['Gene']).intersection(df2['Gene'])
common_genes = pd.DataFrame(common_genes, columns=['Gene'])

In [219]:
common_genes.to_csv('top500.csv', index=False)

In [216]:
len(common_genes)

473

In [7]:
df3

,Gene
0,INPP5D
1,DEFB123
2,NUP210L
3,CHST13
4,E2F6
...,...
7446,DNAJC5B
7447,PGBD5
7448,IFNA10
7449,ANKRD22


In [8]:
df3.to_csv('common_genes_human_antigen.csv', index=False)

In [17]:
df4= pd.read_csv('data/human.csv')
common_genes = set(df3['Gene']).intersection(df4['Gene'])

In [18]:
len(common_genes)

1440

In [3]:
import pandas as pd
df1 = pd.read_csv('pretrained_models/simulate_data/ig_score_changes.csv')
df2 = pd.read_csv('simulate_data/ig_scores.csv')

In [4]:
df2

,Gene,IG Score
0,C1orf141,0.087007
1,PKP1,0.215210
2,RERE,0.054047
3,CSMD2,0.234095
4,HIVEP3,0.106935
...,...,...
27195,ANKRD20A12P,0.051939
27196,MAFIP,0.064963
27197,MGC70870,0.091384
27198,LOC102724728,0.148622


In [5]:
df1

,Gene,IG Score Before Training,IG Score After Training
0,C1orf141,0.268941,0.268941
1,PKP1,0.268941,0.268941
2,RERE,0.268941,0.268919
3,CSMD2,0.268941,0.268941
4,HIVEP3,0.268941,0.268931
...,...,...,...
27195,ANKRD20A12P,0.268941,0.268941
27196,MAFIP,0.268941,0.268941
27197,MGC70870,0.268941,0.268941
27198,LOC102724728,0.268941,0.268941


In [6]:
df3 = pd.merge(df1, df2, on='Gene', how='inner')

In [7]:
df3

,Gene,IG Score Before Training,IG Score After Training,IG Score
0,C1orf141,0.268941,0.268941,0.087007
1,PKP1,0.268941,0.268941,0.215210
2,RERE,0.268941,0.268919,0.054047
3,CSMD2,0.268941,0.268941,0.234095
4,HIVEP3,0.268941,0.268931,0.106935
...,...,...,...,...
27195,ANKRD20A12P,0.268941,0.268941,0.051939
27196,MAFIP,0.268941,0.268941,0.064963
27197,MGC70870,0.268941,0.268941,0.091384
27198,LOC102724728,0.268941,0.268941,0.148622


In [8]:
df3['diff'] = - df3['IG Score'] - df3['IG Score Before Training']
df3['training_diff'] = df3['IG Score After Training']- df3['IG Score Before Training']

In [9]:
df3 = df3[df3['training_diff'] != 0]

In [12]:
df3.to_csv('simulate_data/ig_score_changes_merge.csv', index=False)

In [121]:
df1

,Gene,IG Score Before Training,IG Score After Training,diff
703,ISG15,0.268941,0.268971,0.000030
1098,IFI6,0.268941,0.268990,0.000049
1938,S100A6,0.268941,0.273036,0.004095
4194,EEF1B2,0.268941,0.268978,0.000037
4232,FN1,0.268941,0.268958,0.000017
5841,EIF4A2,0.268941,0.268955,0.000014
6473,SLC34A2,0.268941,0.269752,0.000811
6553,IGFBP7,0.268941,0.268953,0.000011
7704,COX7C,0.268941,0.268953,0.000011
8200,RACK1,0.268941,0.268963,0.000021


In [122]:
common_genes = set(df1['Gene']).intersection(df2['Gene'])

In [123]:
len(common_genes)

45

In [ ]:
adata = sc

In [1]:
import pandas as pd

In [6]:
df1 = pd.read_csv('data/human.csv')
df2 = pd.read_csv('data/filter_gene.csv')

In [10]:
df1

,Gene
0,C1orf141
1,PKP1
2,RERE
3,CSMD2
4,HIVEP3
...,...
27195,ANKRD20A12P
27196,MAFIP
27197,MGC70870
27198,LOC102724728


In [4]:
df2 =df2.drop_duplicates()

In [5]:
df2

,Gene
0,TNFRSF18
1,TNFRSF4
2,ANKRD65
3,VWA1
4,C1orf233
...,...
4892,ZNF706
4893,ZNHIT1
4894,ZRANB2
4895,ZSWIM6


In [8]:
df1_filtered = df1[~df1['Gene'].isin(df2['Gene'])]

In [11]:
df1_filtered.to_csv('data/human_filtered.csv', index=False)